# Titanic — Baseline (Logistic Regression)


## 1. Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

## 2. Загрузка данных

In [2]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
train.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 3. Разведочный анализ (EDA)

In [3]:
train.info()
train.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [4]:
test.info()
test.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    str    
 3   Sex          418 non-null    str    
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    str    
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     str    
 10  Embarked     418 non-null    str    
dtypes: float64(2), int64(4), str(5)
memory usage: 36.1 KB


PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

## 4. Обработка пропусков

In [5]:
train['Age'] = train.groupby(['Pclass','Sex']).Age.transform(lambda x: x.fillna(x.median()))
train['Embarked'] = train.groupby(['Pclass']).Embarked.transform(lambda x: x.fillna(x.mode()[0]))

test['Age'] = test.groupby(['Pclass','Sex']).Age.transform(lambda x: x.fillna(x.median()))
test['Fare'] = test.groupby(['Pclass']).Fare.transform(lambda x: x.fillna(x.median()))

In [6]:
train = train.drop('Cabin', axis=1)
test = test.drop('Cabin', axis=1)

In [7]:
print(f"Пропуски в train:", train.isnull().sum().sum())
print(f"Пропуски в test:", test.isnull().sum().sum())

Пропуски в train: 0
Пропуски в test: 0


## 5. Feature engineering

In [8]:
train.Name.head()

0                              Braund, Mr. Owen Harris
1    Cumings, Mrs. John Bradley (Florence Briggs Th...
2                               Heikkinen, Miss. Laina
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)
4                             Allen, Mr. William Henry
Name: Name, dtype: str

In [9]:
train["Title"] = train.Name.str.extract(r' ([A-ZA-z]+)\.',expand=False)
test["Title"] = test.Name.str.extract(r' ([A-ZA-z]+)\.',expand=False)

train['Title'] = train.Title.replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
test['Title'] = test.Title.replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

In [10]:
print(f"train:",train.Title.value_counts())
print(f"test:",test.Title.value_counts())

train: Title
Mr          517
Miss        185
Mrs         126
Master       40
Dr            7
Rev           6
Major         2
Col           2
Don           1
Lady          1
Sir           1
Capt          1
Countess      1
Jonkheer      1
Name: count, dtype: int64
test: Title
Mr        240
Miss       79
Mrs        72
Master     21
Col         2
Rev         2
Dr          1
Dona        1
Name: count, dtype: int64


In [11]:
rare_titles = train.Title.value_counts()[train.Title.value_counts() < 10].index

train['Title'] = train.Title.replace(rare_titles, 'Rare')
test['Title'] = test.Title.replace(rare_titles, 'Rare')

In [12]:
train.Title.value_counts()

Title
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64

In [13]:
train['FamilySize'] = train.SibSp + train.Parch + 1
test['FamilySize'] = test.SibSp + test.Parch + 1

In [14]:
print(f"train:",train.FamilySize.value_counts())
print(f"test:",test.FamilySize.value_counts())

train: FamilySize
1     537
2     161
3     102
4      29
6      22
5      15
7      12
11      7
8       6
Name: count, dtype: int64
test: FamilySize
1     253
2      74
3      57
4      14
5       7
7       4
11      4
6       3
8       2
Name: count, dtype: int64


## 6. Кодирование категориальных признаков

In [15]:
train = pd.get_dummies(train, columns=['Sex','Embarked','Title'], drop_first=True)
test = pd.get_dummies(test, columns=['Sex','Embarked','Title'], drop_first=True)

In [16]:
train.head()

,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,FamilySize,Sex_male,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,1,0,3,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.2500,2,True,False,True,False,True,False,False
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1,0,PC 17599,71.2833,2,False,False,False,False,False,True,False
2,3,1,3,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.9250,1,False,False,True,True,False,False,False
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1000,2,False,False,True,False,False,True,False
4,5,0,3,"Allen, Mr. William Henry",35.0,0,0,373450,8.0500,1,True,False,True,False,True,False,False


## 7. Обучение baseline-модели

In [20]:
drop_cols = ['Name', 'Ticket', 'PassengerId', 'SibSp', 'Parch']
train = train.drop(columns=drop_cols)

test_passenger_ids = test['PassengerId']
test = test.drop(columns=drop_cols)

In [21]:
X = train.drop('Survived',axis=1)
y = train['Survived']

model = LogisticRegression(max_iter=1000)
model.fit(X,y)

print("Модель обучена")
print(model.coef_)

Модель обучена
[[-1.10206227 -0.03581145  0.00338357 -0.39292201 -0.83396009 -0.11034112
  -0.37616148 -0.45320773 -2.37842038  0.40840444 -1.68503613]]


In [22]:
list(zip(X.columns, model.coef_[0]))

[('Pclass', np.float64(-1.1020622742813642)),
 ('Age', np.float64(-0.0358114477107594)),
 ('Fare', np.float64(0.003383573664741088)),
 ('FamilySize', np.float64(-0.3929220089400291)),
 ('Sex_male', np.float64(-0.8339600879606093)),
 ('Embarked_Q', np.float64(-0.11034112007606887)),
 ('Embarked_S', np.float64(-0.37616148205500033)),
 ('Title_Miss', np.float64(-0.4532077343797245)),
 ('Title_Mr', np.float64(-2.3784203816297933)),
 ('Title_Mrs', np.float64(0.4084044355856103)),
 ('Title_Rare', np.float64(-1.685036125961082))]

## 8. Кросс-валидация

In [24]:
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print("Score по каждому фолду:", scores)
print("Средний accuracy:", scores.mean())
print("Стандартное отклонение:", scores.std())

InvalidParameterError: The 'scoring' parameter of cross_val_score must be a str among {'neg_mean_absolute_percentage_error', 'jaccard_micro', 'precision_macro', 'neg_brier_score', 'jaccard_weighted', 'explained_variance', 'completeness_score', 'd2_brier_score', 'v_measure_score', 'balanced_accuracy', 'f1_macro', 'normalized_mutual_info_score', 'accuracy', 'mutual_info_score', 'recall', 'roc_auc_ovo', 'neg_mean_squared_error', 'roc_auc_ovr_weighted', 'recall_micro', 'adjusted_mutual_info_score', 'fowlkes_mallows_score', 'roc_auc', 'jaccard_macro', 'recall_samples', 'neg_mean_squared_log_error', 'neg_median_absolute_error', 'f1_weighted', 'recall_weighted', 'roc_auc_ovo_weighted', 'precision', 'jaccard', 'roc_auc_ovr', 'average_precision', 'jaccard_samples', 'd2_log_loss_score', 'precision_samples', 'r2', 'neg_mean_poisson_deviance', 'neg_mean_gamma_deviance', 'top_k_accuracy', 'precision_micro', 'rand_score', 'neg_log_loss', 'f1_samples', 'matthews_corrcoef', 'neg_negative_likelihood_ra

## 9. Предсказание на test и сабмит

In [19]:
# TODO: predict + сохранить submission.csv


## 10. Выводы

- Что сработало
- Что можно улучшить